<a href="https://colab.research.google.com/github/Prashantgitpera/-wfm-machine-learning/blob/main/Time_series_forecasting_in_ML_(ARIMA%2C_Holt_Winters).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
!pip install pmdarima
from pmdarima import auto_arima

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Forecasting/ Time series forecasting in ML (ARIMA, Holt-Winters)/Raw data.xlsx')

In [ ]:
def convert_quarter_to_date(quarter_str):# Convert 'Period' to datetime for better time series handling
    q, year = quarter_str.split('-') # Assuming Q1 is Jan 1, Q2 is Apr 1, Q3 is Jul 1, Q4 is Oct 1
    year = int(year)
    if q == 'Q1': return datetime(year, 1, 1)
    elif q == 'Q2': return datetime(year, 4, 1)
    elif q == 'Q3': return datetime(year, 7, 1)
    elif q == 'Q4': return datetime(year, 10, 1)

df['Date'] = df['Period'].apply(convert_quarter_to_date)
df.set_index('Date', inplace=True)
df = df.drop(columns=['Period'])

display(df.head())

In [ ]:
df.head(12)

In [ ]:
plt.figure (figsize=(10,5))
plt.plot(df.index, df['Volume'], marker='o',label = 'Volume')
plt.title('offered volume')
plt.legend()
plt.xlabel('Date')
plt.ylabel('Volume')
# plt.xticks(ticks=range(len(df.index)), labels=df.index.strftime('%Y-%m'), rotation=45, ha='right') # Removed explicit xticks for auto-formatting
plt.grid(True)
plt.show()

In [ ]:
decomposed = seasonal_decompose(df['Volume'], model='additive', period=12)
fig = decomposed.plot()
fig.set_figheight(6)
fig.set_figwidth(12)
plt.show()

In [ ]:
def check_stationarity(timeseries):
    result = adfuller(timeseries)
    print('Augmented Dickey-fuller Test')
    print('ADF Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print('\t%s: %.3f' % (key, value))
    return result[1]< 0.05

is_stationary = check_stationarity(df['Volume'])
print('Is the time series stationary?', is_stationary)

In [ ]:
auto_arima_model = auto_arima(df['Volume'],
                              start_p=1, start_q=1,
                              max_p=3, max_q=3,d=None,
                              seasonal=False, m=12,
                              trace=True, error_action='ignore',
                              suppress_warnings=True,
                              stepwise=True)

In [ ]:
# Generate forecasts for the next 8 periods
n_periods = 8
forecast, conf_int = auto_arima_model.predict(n_periods=n_periods, return_conf_int=True)

# Create a date index for the forecast
last_date = df.index[-1]
forecast_index = pd.date_range(start=last_date, periods=n_periods + 1, freq='QS')[1:] # 'QS' for Quarter Start

# Create a Series for the forecast
forecast_series = pd.Series(forecast, index=forecast_index)
lower_series = pd.Series(conf_int[:, 0], index=forecast_index)
upper_series = pd.Series(conf_int[:, 1], index=forecast_index)

# Plot the results
plt.figure(figsize=(15, 7))
plt.plot(df['Volume'], marker='o',label='Historical Volume')
plt.plot(forecast_series, color='red',marker='o',label='Forecasted Volume')
plt.fill_between(lower_series.index, lower_series, upper_series, color='grey', alpha=0.3, label='Confidence Interval')
plt.title('Volume Forecast vs. Historical Data')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.legend()
plt.grid(True)
plt.show()

### Model Evaluation: Calculating MAE, RMSE, MAPE, and FA

To properly evaluate the model, we need to split our historical data into a training set and a testing set. We will train the `auto_arima` model on the training data, forecast values for the test period, and then compare these forecasts against the actual values in the test set.

In [ ]:
# Split data into training and testing sets
train_size = int(len(df) * 0.8)
train_data, test_data = df.iloc[:train_size], df.iloc[train_size:]

print(f"Training data points: {len(train_data)}")
print(f"Testing data points: {len(test_data)}")

# Retrain auto_arima model on training data
retrained_auto_arima_model = auto_arima(train_data['Volume'],
                                        start_p=1, start_q=1,
                                        max_p=3, max_q=3, d=None,
                                        seasonal=False, m=12,
                                        trace=True, error_action='ignore',
                                        suppress_warnings=True,
                                        stepwise=True)

# Make predictions on the test set
n_periods_test = len(test_data)
forecast_test, conf_int_test = retrained_auto_arima_model.predict(n_periods=n_periods_test, return_conf_int=True)

# Create a Series for the test forecast
forecast_test_series = pd.Series(forecast_test, index=test_data.index)

# Plot the training data, actual test data, and forecasted test data
plt.figure(figsize=(15, 7))
plt.plot(train_data['Volume'], label='Training Data')
plt.plot(test_data['Volume'], label='Actual Test Data', color='orange')
plt.plot(forecast_test_series, color='red', label='Forecasted Test Data')
plt.fill_between(test_data.index, conf_int_test[:, 0], conf_int_test[:, 1], color='pink', alpha=0.3, label='Confidence Interval')
plt.title('ARIMA Model: Training, Actual Test, and Forecasted Test Data')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.legend()
plt.grid(True)
plt.show()

Now, let's calculate the requested error metrics: MAE, RMSE, MAPE, and FA.

In [ ]:
# Calculate error metrics

# Mean Absolute Error (MAE)
mae = np.mean(np.abs(forecast_test_series - test_data['Volume']))

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(test_data['Volume'], forecast_test_series))

# Mean Absolute Percentage Error (MAPE)
# Avoid division by zero: replace 0 in actuals with a small epsilon if present
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Replace zero actuals with a small number to avoid division by zero
    return np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, np.finfo(float).eps))) * 100

mape = mean_absolute_percentage_error(test_data['Volume'], forecast_test_series)

# Forecasting Accuracy (FA) - often defined as 100 - MAPE
fa = 100 - mape

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Forecasting Accuracy (FA): {fa:.2f}%")

In [ ]:
display(forecast_series)

In [ ]:
!pwd

In [ ]:
forecast_series.to_excel('/content/forecasted_values.xlsx')
print("Forecasted values saved to '/content/forecasted_values.xlsx'")

In [ ]:
import shutil

source_path = '/content/forecasted_values.xlsx'
destination_path = '/content/drive/MyDrive/Colab Notebooks/Forecasting/ Time series forecasting in ML (ARIMA, Holt-Winters)/forecasted_values.xlsx'

shutil.copy(source_path, destination_path)
print(f"File copied to: {destination_path}")